In [2]:
import torch
from datasets import Dataset, DatasetDict
from typing import Tuple, List


def moral_machine_dpo(
    train_pt_path: str,
    test_pt_path: str,
    output_path: str = "moral_machine_dpo_dataset",
    push_to_hub: bool = False,
    hub_name: str = None
) -> DatasetDict:
    """
    Create a HuggingFace dataset for DPO training from Moral Machine .pt files.
    
    Args:
        train_pt_path: Path to training .pt file containing list of (question, answer) tuples
        test_pt_path: Path to test .pt file containing list of (question, answer) tuples
        output_path: Local path to save the dataset (default: "moral_machine_dpo_dataset")
        push_to_hub: Whether to push to HuggingFace Hub (default: False)
        hub_name: Repository name on HuggingFace Hub (required if push_to_hub=True)
    
    Returns:
        DatasetDict containing 'train' and 'test' splits with DPO format:
        - prompt: The question/scenario
        - chosen: The preferred response
        - rejected: The complement response (STAY <-> SWERVE)
    """
    
    def flip_answer(answer: str) -> str:
        """Flip STAY to SWERVE and vice versa."""
        answer_upper = answer.strip().upper()
        if answer_upper == "STAY":
            return "SWERVE"
        elif answer_upper == "SWERVE":
            return "STAY"
        else:
            raise ValueError(f"Answer must be 'STAY' or 'SWERVE', got: {answer}")
    
    def process_split(data: List[Tuple[str, str]]) -> Dataset:
        """Convert list of tuples to DPO-formatted dataset."""
        processed_data = {
            "prompt": [],
            "chosen": [],
            "rejected": []
        }
        
        for question, answer in data:
            processed_data["prompt"].append(question)
            processed_data["chosen"].append(answer)
            processed_data["rejected"].append(flip_answer(answer))
        
        return Dataset.from_dict(processed_data)
    
    # Load the .pt files
    print(f"Loading training data from {train_pt_path}...")
    train_data = torch.load(train_pt_path)
    
    print(f"Loading test data from {test_pt_path}...")
    test_data = torch.load(test_pt_path)
    
    # Validate data format
    if not isinstance(train_data, list) or not isinstance(test_data, list):
        raise ValueError("Both .pt files must contain lists of tuples")
    
    if len(train_data) > 0:
        if not isinstance(train_data[0], tuple) or len(train_data[0]) != 2:
            raise ValueError("Train data must be list of (question, answer) tuples")
    
    if len(test_data) > 0:
        if not isinstance(test_data[0], tuple) or len(test_data[0]) != 2:
            raise ValueError("Test data must be list of (question, answer) tuples")
    
    print(f"Processing {len(train_data)} training examples...")
    train_dataset = process_split(train_data)
    
    print(f"Processing {len(test_data)} test examples...")
    test_dataset = process_split(test_data)
    
    # Create DatasetDict
    dataset_dict = DatasetDict({
        "train": train_dataset,
        "test": test_dataset
    })
    
    # Save locally
    print(f"Saving dataset to {output_path}...")
    dataset_dict.save_to_disk(output_path)
    
    # Optionally push to hub
    if push_to_hub:
        if hub_name is None:
            raise ValueError("hub_name must be provided when push_to_hub=True")
        print(f"Pushing dataset to HuggingFace Hub: {hub_name}...")
        dataset_dict.push_to_hub(hub_name)
    
    print("Dataset creation complete!")
    print(f"Train size: {len(train_dataset)}")
    print(f"Test size: {len(test_dataset)}")
    print(f"\nDataset structure:")
    print(dataset_dict)
    
    return dataset_dict

In [3]:
dataset = moral_machine_dpo("moral_data_train.pt","moral_data_test.pt", "moral_choices")

Loading training data from moral_data_train.pt...
Loading test data from moral_data_test.pt...
Processing 250000 training examples...
Processing 50000 test examples...
Saving dataset to moral_choices...


Saving the dataset (0/1 shards):   0%|          | 0/250000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/50000 [00:00<?, ? examples/s]

Dataset creation complete!
Train size: 250000
Test size: 50000

Dataset structure:
DatasetDict({
    train: Dataset({
        features: ['prompt', 'chosen', 'rejected'],
        num_rows: 250000
    })
    test: Dataset({
        features: ['prompt', 'chosen', 'rejected'],
        num_rows: 50000
    })
})
